# HW 10 starter, the P-Card data

This notebook loads the Homework 10 dataset and leaves you with one starter query pointed at each of the four menu methods. Run it top to bottom, then pick a direction and start asking your own questions.

**The data.** Every purchase-card transaction made by Oklahoma state agencies across five fiscal years, FY2022 through FY2026: 2,055,905 rows from roughly 120 agencies. A purchase card, or P-Card, is a government credit card for routine buying, so this file is the day-to-day spending of a state government: agencies, merchants, amounts, and dates.

**Attribution.** Public record, published by the State of Oklahoma at [data.ok.gov](https://data.ok.gov) under a CC BY license. The cardholder-name columns were dropped when the course file was built; the analysis columns are as published.

**A caution before you trust any total.** The file is exactly what the state publishes, so it carries real warts: 69,430 negative amounts (refunds and credits), 1,423 rows with a blank amount, and one month whose dates shipped blank. Code that ignores them runs fine and answers slightly wrong.

The assignment, the four-method menu, and the challenge-prompt seeds are on [the HW 10 page](https://seanmccaman.com/acctg5150/2026-summer/week-10/hw/hw_10.html).

In [ ]:
%pip install -q duckdb

In [ ]:
import duckdb

# The data stays on the web. DuckDB reads just the bytes each query needs,
# so you can query 2 million rows without downloading the file first.
URL = "https://raw.githubusercontent.com/sean-mccaman/acctg5150-090/main/2026-summer/week-10/pcard.parquet"

duckdb.sql(f"SELECT COUNT(*) AS rows, MIN(FISCAL_YEAR) AS first_fy, MAX(FISCAL_YEAR) AS last_fy FROM '{URL}'").df()

In [ ]:
# Fallback for locked-down machines. Some corporate and lab networks block
# direct web reads. If the cell above errored, uncomment the two lines below:
# they download the file once (about 35 MB) with plain urllib, and every query
# after this points at whichever copy SRC names.
import urllib.request

SRC = URL
# urllib.request.urlretrieve(URL, "pcard.parquet")
# SRC = "pcard.parquet"

duckdb.sql(f"SELECT COUNT(*) AS rows FROM '{SRC}'").df()

In [ ]:
# Time-series direction: monthly spend by fiscal year, on the state's posting
# calendar. Judge a month against its own year, and if you rebuild the months
# from TRANSACTION_DATE instead, one month nearly vanishes, which is a clue
# about the file rather than a bug in your code.
monthly = duckdb.sql(f"""
    SELECT FISCAL_YEAR, CALENDAR_YEAR, CALENDAR_MONTH,
           ROUND(SUM(AMOUNT)) AS spend_dollars, COUNT(*) AS txns
    FROM '{SRC}'
    WHERE AMOUNT > 0
    GROUP BY ALL
    ORDER BY CALENDAR_YEAR, CALENDAR_MONTH
""").df()
monthly

In [ ]:
# Anomaly direction: the exact amounts just around the $5,000 purchase limit.
# Ask why some values repeat this often, and what a fair comparison would be.
near_limit = duckdb.sql(f"""
    SELECT AMOUNT, COUNT(*) AS txns
    FROM '{SRC}'
    WHERE AMOUNT BETWEEN 4900 AND 5100
    GROUP BY AMOUNT
    ORDER BY txns DESC
    LIMIT 15
""").df()
near_limit

In [ ]:
# Benford direction: the first significant digit of every positive amount.
# Benford's curve starts near 30.1% for 1s and falls to 4.6% for 9s, and the
# menu page says why the first-TWO-digit version is where the story lives.
first_digit = duckdb.sql(f"""
    SELECT regexp_extract(CAST(AMOUNT AS VARCHAR), '[1-9]') AS first_digit,
           COUNT(*) AS txns,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM '{SRC}'
    WHERE AMOUNT > 0
    GROUP BY first_digit
    ORDER BY first_digit
""").df()
first_digit

In [ ]:
# Clustering direction: one behavioral row per agency. Key on AGENCYNBR,
# because some agencies changed names mid-window and keying on the name
# silently splits them into duplicates.
agencies = duckdb.sql(f"""
    SELECT AGENCYNBR,
           arg_max(AGENCYNAME, TRANSACTION_DATE) AS agency,
           COUNT(*) AS txns,
           ROUND(SUM(AMOUNT)) AS total_spend,
           ROUND(AVG(AMOUNT), 2) AS avg_txn
    FROM '{SRC}'
    WHERE AMOUNT > 0
    GROUP BY AGENCYNBR
    ORDER BY total_spend DESC
    LIMIT 20
""").df()
agencies

## Where to go from here

Pick the direction that interests you and deepen its query: more columns, a tighter filter, a per-year split. When you land on a pattern you can state with numbers, the discovery half is done and the graded half starts, which is challenging the finding and verifying it across more than one model. The workflow, the challenge-prompt seeds, and what to submit are on [the HW 10 page](https://seanmccaman.com/acctg5150/2026-summer/week-10/hw/hw_10.html).